# Day 14 · Lab 3/4: joins and window functions, and how Spark executes them

35 of Zillacode's 52 problems join tables, and most of the Hard ones need a window function. In real jobs, joins and windows are also where most of the time goes, so today covers both tracks in depth:
1. Join types, and what decides the number of rows
2. 🔬 How Spark executes a join: five algorithms, how Spark picks one, hints, and AQE's switch at run time
3. Same column names, one table in two roles, self-joins
4. Joining on a range, and cross joins
5. 🔬 Skew: one key with too many rows, and three ways to fix it
6. Window functions: a group's value on every row, and how a window runs
7. Ranking: `row_number`, `rank`, `dense_rank`, top N and latest row
8. Previous and next rows, running totals and the frame trap

In Zillacode's topic list this is *Complex Joins* and *Windows Functions*. On the optimization side it covers theory Days 5 and 6: join selection, broadcast, shuffle hash and sort-merge joins, hints, AQE's join switch and skew handling, and salting.

**New today: big tables.** The third setup cell generates 10 million sales (about 160 MB of Parquet, once, in about 15 seconds), because on 16 rows every join takes 0.1 seconds and nothing can be measured.

## How to work
* Run the four **Setup** cells first, then the cell that prints the small tables.
* **Lessons**: run every example cell (its output is already shown under it), then change it and run it again until you can predict the result. On the big tables your times will differ from the stored ones; the plans, stages, tasks and bytes should not.
* Exercises come right after each lesson, as on Day 13: **`etl(...)`** in Zillacode's format, **🔍 Read the plan** (answer as a comment in the cell), and **⚡ Make it faster** (the expected output *and* the target under it).
* Keep the **Spark UI** open (normally http://localhost:4040): the *SQL / DataFrame* tab shows each join's algorithm and row counts, the *Stages* tab the slowest task of every stage.
* The answers are collapsed at the very end of the notebook. Open one only after you have tried.
* Today: about 1 hour 50 minutes of lessons and 2 hours of exercises (14 `etl`, 2 🔍 and 3 ⚡).

**Setup (once):** Java 17 or 21 and Python 3.10 or newer. In a new folder: `python3.12 -m venv .venv`, `source .venv/bin/activate`, `pip install pyspark==4.2.0 jupyterlab`, then `jupyter lab` and open this notebook. The notebook writes Parquet files into a `data/` folder next to it (about 160 MB; delete the folder when you're done with Day 15).

## The data
The small shop tables of Days 12 and 13 (saved as Parquet in `data/shop/`), for the `etl` exercises, and three big generated tables for the 🔬 lessons.

* **customers**: *customer_id*, *name*, *email*, *city*, *signup_date*
* **products**: *product_id*, *product_name*, *category*, *price* (dollars for one item)
* **orders**: *order_id*, *customer_id*, *product_id*, *quantity*, *order_date*, *status* (`delivered`, `shipped` or `cancelled`)
* **reviews**: *review_id*, *product_id*, *customer_id*, *review_date* (text like `15/01/2024`), *rating* (1 to 5), *comment* (messy text), *tags* (text like `fast delivery;good price`)
* **signups_raw**: sign-ups exactly as a web form sent them: *signup_id*, *full_name*, *email*, *phone*, *signup_date* (text in three formats), *amount* (text), *prefs* (JSON text)
* **customers_new**: the customers table one week later
* **promotions**: *promo_id*, *category*, *start_date*, *end_date* (both days included), *discount_pct*
* **events**: website visits: *customer_id*, *event_time* (a timestamp), *page*
* **referrals**: *referrer_id* (the customer who invited someone), *referred_id* (the customer who joined), *referral_date*

The data has the traps that Zillacode's hidden tests like to use: order 1011 is a guest checkout (no *customer_id*); customer 6 has no email and no city; customer 7 has never ordered; product 107 has never been sold; order 1016 is for product 108, which is not in *products*; review 6 has no tags; two events were loaded twice; referral 5 points to customer 9, who doesn't exist.

**The big tables** (third setup cell), for the 🔬 lessons:
* **sales**: 10 million rows: *sale_id*, *customer_id*, *product_id*, *store_id* (1 to 200), *quantity* (1 to 5), *sale_date* (2023-01-01 to 2024-12-31), *channel* (`web`, `app` or `store`). About 140 MB.
* **customers_big**: 2 million customers: *customer_id*, *name*, *city* (10 cities), *segment* (`business` or `private`). About 18 MB, just above Spark's 10 MB broadcast limit, on purpose.
* **products_big**: 50,000 products: *product_id*, *product_name*, *category* (10 categories), *price*. Under 1 MB.

**Customer 1 is a reseller who makes 30% of all sales** (about 3 million rows). Every other customer has a handful. Lesson 5 is about what this does to a join.

## Setup
Run these cells once: Spark and the small tables, `measure()` (Day 13, Lesson 1), and the big tables, which are generated only if `data/big/` doesn't exist yet.

In [ ]:
# Setup: starts Spark, creates the tables and saves them as Parquet files. Run it once.
import logging, os, sys
from datetime import date
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)   # Spark's Python workers use this notebook's Python
from pyspark.sql import SparkSession, functions as F, Window as W   # the same names as in Zillacode's editor

spark = (SparkSession.builder
         .master("local[4]")                               # 4 cores: up to 4 tasks run at the same time
         .appName("spark-lab-14")
         .config("spark.driver.memory", "2g")              # the default 1 GB is tight for 10 million rows
         .config("spark.sql.ansi.enabled", "false")        # behave like Zillacode (Spark 3): bad text gives null, not an error
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.driver.host", "localhost")         # run on this machine only; the Spark UI is at http://localhost:4040
         .config("spark.sql.maxMetadataStringLength", "1000")   # explain() prints long filter and column lists in full
         .config("spark.hadoop.parquet.hadoop.vectored.io.enabled", "false")   # Spark 4.2: without this, the UI's "Input" misses most bytes read
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.log.level", "FATAL")               # hide Spark's log messages
         # Spark 4.2.0 bug (SPARK-60294): Python UDFs on Parquet can hang without this
         .config("spark.sql.execution.arrow.pythonUDF.columnarInput.enabled", "false")
         .getOrCreate())
for name in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
    logging.getLogger(name).setLevel(logging.CRITICAL)        # errors still show, without an extra JSON log

customers = spark.createDataFrame([
    # customer_id, name, email, city, signup_date
    (1, "Aziz Karimov",     "aziz@mail.uz",     "Tashkent", date(2024, 1, 5)),
    (2, "Bella Schmidt",    "bella@web.de",     "Berlin",   date(2024, 1, 20)),
    (3, "Carlos Ruiz",      "carlos@correo.es", "Madrid",   date(2024, 2, 2)),
    (4, "Dilnoza Rahimova", "dilnoza@mail.uz",  "Tashkent", date(2024, 2, 14)),
    (5, "Emma Wilson",      "emma@post.co.uk",  "London",   date(2024, 3, 1)),
    (6, "Farid Aliyev",     None,               None,       date(2024, 3, 10)),
    (7, "Grace Lee",        "grace@web.de",     "Berlin",   date(2024, 3, 15)),
], "customer_id INT, name STRING, email STRING, city STRING, signup_date DATE")

products = spark.createDataFrame([
    # product_id, product_name, category, price (dollars for one item)
    (101, "Laptop",       "Electronics", 1200.0),
    (102, "Headphones",   "Electronics",  150.0),
    (103, "Coffee Maker", "Kitchen",       85.0),
    (104, "Blender",      "Kitchen",       60.0),
    (105, "Novel",        "Books",         15.0),
    (106, "Cookbook",     "Books",         25.0),
    (107, "Desk Lamp",    "Home",          40.0),
], "product_id INT, product_name STRING, category STRING, price DOUBLE")

orders = spark.createDataFrame([
    # order_id, customer_id, product_id, quantity, order_date, status
    (1001, 1,    101, 1, date(2024, 1, 10), "delivered"),
    (1002, 1,    105, 3, date(2024, 1, 10), "delivered"),
    (1003, 2,    102, 2, date(2024, 1, 25), "delivered"),
    (1004, 3,    103, 1, date(2024, 2, 3),  "cancelled"),
    (1005, 2,    106, 1, date(2024, 2, 5),  "delivered"),
    (1006, 4,    104, 2, date(2024, 2, 15), "shipped"),
    (1007, 1,    102, 1, date(2024, 2, 20), "delivered"),
    (1008, 5,    101, 1, date(2024, 3, 2),  "shipped"),
    (1009, 3,    105, 4, date(2024, 3, 5),  "delivered"),
    (1010, 4,    106, 2, date(2024, 3, 8),  "cancelled"),
    (1011, None, 104, 1, date(2024, 3, 9),  "delivered"),
    (1012, 6,    103, 2, date(2024, 3, 12), "shipped"),
    (1013, 2,    101, 1, date(2024, 3, 18), "delivered"),
    (1014, 5,    105, 5, date(2024, 3, 20), "delivered"),
    (1015, 1,    103, 1, date(2024, 3, 25), "shipped"),
    (1016, 3,    108, 1, date(2024, 3, 28), "delivered"),
], "order_id INT, customer_id INT, product_id INT, quantity INT, order_date DATE, status STRING")

reviews = spark.createDataFrame([
    # review_id, product_id, customer_id, review_date (text), rating, comment, tags (text)
    (1, 101, 1, "15/01/2024", 5, "  Great laptop, very FAST!  ",    "fast delivery;good price"),
    (2, 105, 1, "20/01/2024", 4, "Nice story.",                     "good price"),
    (3, 102, 2, "02/02/2024", 3, "Sound is ok, delivery was slow",  "slow delivery"),
    (4, 103, 3, "10/02/2024", 1, "Broke after one week!!",          "broken;slow delivery"),
    (5, 101, 5, "09/03/2024", 5, "Fast and powerful",               "fast delivery"),
    (6, 105, 3, "12/03/2024", 2, "Too long, boring",                None),
    (7, 106, 2, "25/03/2024", 4, "Useful recipes",                  "good price;gift"),
    (8, 103, 6, "30/03/2024", 4, "Good coffee, slow start",         "gift"),
], "review_id INT, product_id INT, customer_id INT, review_date STRING, rating INT, comment STRING, tags STRING")

signups_raw = spark.createDataFrame([
    # signup_id, full_name, email, phone, signup_date (text, three formats), amount (text), prefs (JSON text)
    (1, "  aziz   KARIMOV ", "Aziz@Mail.UZ ",      "+998 90 123-45-67", "2024-01-05", "12.50", '{"newsletter": true, "lang": "uz"}'),
    (2, "bella schmidt",     "bella@web.de",       "0049 151 2345678",  "20/01/2024", "abc",   '{"newsletter": false, "lang": "de"}'),
    (3, "Carlos  Ruiz",      "",                   "(34) 612 345 678",  "2024/02/02", "7",     '{"lang": "es"}'),
    (4, "DILNOZA rahimova",  "dilnoza@mail.uz",    "90-765-43-21",      "14/02/2024", "",      None),
    (5, "Emma Wilson",       "emma[at]post.co.uk", "07700 900123",      "2024-03-01", "3.75",  '{"newsletter": true}'),
    (6, "aziz karimov",      "aziz@mail.uz",       "+998901234567",     "2024-03-02", "1.00",  '{"newsletter": false, "lang": "uz"}'),
], "signup_id INT, full_name STRING, email STRING, phone STRING, signup_date STRING, amount STRING, prefs STRING")

customers_new = spark.createDataFrame([
    # the customers table one week later: 2 moved, 6 has an email now, 7 is gone, 8 is new
    (1, "Aziz Karimov",     "aziz@mail.uz",     "Tashkent", date(2024, 1, 5)),
    (2, "Bella Schmidt",    "bella@web.de",     "Munich",   date(2024, 1, 20)),
    (3, "Carlos Ruiz",      "carlos@correo.es", "Madrid",   date(2024, 2, 2)),
    (4, "Dilnoza Rahimova", "dilnoza@mail.uz",  "Tashkent", date(2024, 2, 14)),
    (5, "Emma Wilson",      "emma@post.co.uk",  "London",   date(2024, 3, 1)),
    (6, "Farid Aliyev",     "farid@mail.az",    None,       date(2024, 3, 10)),
    (8, "Hana Sato",        "hana@mail.jp",     "Osaka",    date(2024, 4, 2)),
], customers.schema)

promotions = spark.createDataFrame([
    # promo_id, category, start_date, end_date (both days included), discount_pct
    ("WINTER",   "Electronics", date(2024, 1, 1),  date(2024, 1, 31), 10),
    ("BOOKWEEK", "Books",       date(2024, 3, 1),  date(2024, 3, 10), 20),
    ("SPRING",   "Kitchen",     date(2024, 3, 1),  date(2024, 3, 31), 15),
    ("FLASH",    "Electronics", date(2024, 3, 15), date(2024, 3, 20), 5),
], "promo_id STRING, category STRING, start_date DATE, end_date DATE, discount_pct INT")

events = spark.createDataFrame([
    # customer_id, event_time, page (two rows were loaded twice)
    (1, "2024-03-01 09:00:00", "home"),
    (1, "2024-03-01 09:10:00", "product"),
    (1, "2024-03-01 09:10:00", "product"),
    (1, "2024-03-01 11:00:00", "home"),
    (1, "2024-03-02 10:00:00", "cart"),
    (1, "2024-03-03 10:30:00", "home"),
    (1, "2024-03-05 08:00:00", "home"),
    (2, "2024-03-01 20:00:00", "home"),
    (2, "2024-03-01 20:20:00", "product"),
    (2, "2024-03-01 20:45:00", "cart"),
    (2, "2024-03-04 18:00:00", "home"),
    (2, "2024-03-05 18:00:00", "home"),
    (2, "2024-03-05 18:00:00", "home"),
    (3, "2024-03-02 12:00:00", "product"),
    (3, "2024-03-03 12:00:00", "product"),
    (3, "2024-03-04 12:00:00", "cart"),
], "customer_id INT, event_time STRING, page STRING").withColumn("event_time", F.to_timestamp("event_time"))

referrals = spark.createDataFrame([
    # referrer_id (the customer who invited), referred_id (the customer who joined), referral_date
    (1, 4, date(2024, 2, 10)),
    (4, 5, date(2024, 2, 25)),
    (1, 6, date(2024, 3, 8)),
    (2, 7, date(2024, 3, 12)),
    (2, 9, date(2024, 3, 30)),
], "referrer_id INT, referred_id INT, referral_date DATE")

# Save every table as Parquet and read it back. In-memory tables would show "Scan ExistingRDD" in plans;
# Parquet files show what real jobs read: "FileScan parquet" with the columns and filters it reads.
for name, df in [("customers", customers), ("products", products), ("orders", orders), ("reviews", reviews),
                 ("signups_raw", signups_raw), ("customers_new", customers_new), ("promotions", promotions),
                 ("events", events), ("referrals", referrals)]:
    df.coalesce(1).write.mode("overwrite").parquet(f"data/shop/{name}")   # one file per table
customers = spark.read.parquet("data/shop/customers")
products = spark.read.parquet("data/shop/products")
orders = spark.read.parquet("data/shop/orders")
reviews = spark.read.parquet("data/shop/reviews")
signups_raw = spark.read.parquet("data/shop/signups_raw")
customers_new = spark.read.parquet("data/shop/customers_new")
promotions = spark.read.parquet("data/shop/promotions")
events = spark.read.parquet("data/shop/events")
referrals = spark.read.parquet("data/shop/referrals")

print("Spark", spark.version, "is ready. Spark UI:", spark.sparkContext.uiWebUrl)
print("Tables: customers, products, orders, reviews, signups_raw, customers_new, promotions, events, referrals")

Spark 4.2.0 is ready. Spark UI: http://localhost:4040
Tables: customers, products, orders, reviews, signups_raw, customers_new, promotions, events, referrals


In [ ]:
# measure(): run a query and print what it cost. These are the numbers the Spark UI shows.
import json, time, urllib.request

def ui(path):
    """Read one page of the Spark UI's REST API (the data behind http://localhost:4040)."""
    sc = spark.sparkContext
    with urllib.request.urlopen(f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}/{path}") as response:
        return json.load(response)

def size(n):
    for unit, k in (("GB", 1e9), ("MB", 1e6), ("KB", 1e3)):
        if n >= k:
            return f"{n / k:.1f} {unit}"
    return f"{n} B"

def measure(query, label="", stages=False, plan=False):
    """Run `query` to the end, then print: time, jobs, stages, tasks, bytes read from files, shuffled and spilled.

    query:  a DataFrame (it runs with the "noop" writer: every task runs, nothing is kept or shown),
            or a function without arguments, like  lambda: df.count()
    stages: also print one line per stage: its tasks, the median and the slowest task, its shuffle
    plan:   also print the final plan of the query, after AQE's changes (the plan in the UI's SQL tab)
    """
    sc = spark.sparkContext
    group = f"measure-{time.time_ns()}"
    sc.setJobGroup(group, label or "measure")
    start = time.perf_counter()
    try:
        if callable(query):
            query()
        else:
            query.write.format("noop").mode("overwrite").save()
    finally:
        sc.setLocalProperty("spark.jobGroup.id", None)
    seconds = time.perf_counter() - start
    tracker = sc.statusTracker()
    jobs = sorted(tracker.getJobIdsForGroup(group))
    stage_ids = sorted({s for j in jobs for s in tracker.getJobInfo(j).stageIds})
    for _ in range(100):                     # the UI records a stage a moment after it ends
        info = [ui(f"stages/{s}")[0] for s in stage_ids]
        if all(s["status"] in ("COMPLETE", "SKIPPED", "FAILED") for s in info):
            break
        time.sleep(0.05)
    ran = [s for s in info if s["status"] == "COMPLETE"]       # "SKIPPED": output reused, did not run
    total = lambda key: sum(s[key] for s in ran)
    print(f"{label + ': ' if label else ''}{seconds:.2f} s | {len(jobs)} jobs, {len(ran)} stages, "
          f"{total('numCompleteTasks')} tasks | read {size(total('inputBytes'))} | "
          f"shuffle {size(total('shuffleWriteBytes'))} | spill {size(total('diskBytesSpilled'))}")
    if stages:
        for s in ran:
            t = ui(f"stages/{s['stageId']}/{s['attemptId']}/taskSummary?quantiles=0.5,1.0")["executorRunTime"]
            print(f"   stage {s['stageId']:>3}: {s['numCompleteTasks']:>3} tasks, median {t[0] / 1000:.2f} s, "
                  f"max {t[1] / 1000:.2f} s | read {size(s['inputBytes'])} | "
                  f"shuffle read {size(s['shuffleReadBytes'])}, write {size(s['shuffleWriteBytes'])}")
    if plan:
        runs = [e for e in ui("sql?offset=0&length=1000000") if set(e["successJobIds"]) & set(jobs)]
        tree = ui(f"sql/{runs[-1]['id']}?planDescription=true")["planDescription"].split("\n\n")[0]
        print(tree.replace(", Statistics(sizeInBytes=8.0 EiB)", ""))

print("measure() is ready")

measure() is ready


In [ ]:
# The big tables: generated once into data/big/ (about 15 seconds), then only read
import os
if not os.path.exists("data/big/sales/_SUCCESS"):
    spark.range(1, 10_000_001, numPartitions=8).select(
        F.col("id").alias("sale_id"),
        # customer 1 (a reseller) gets 30% of the sales; the rest are spread over customers 2 to 2,000,000
        F.when(F.rand(1) < 0.3, 1).otherwise((F.rand(2) * 1_999_999).cast("int") + 2).alias("customer_id"),
        ((F.rand(3) * 50_000).cast("int") + 1).alias("product_id"),
        ((F.rand(4) * 200).cast("int") + 1).alias("store_id"),
        ((F.rand(5) * 5).cast("int") + 1).alias("quantity"),
        F.date_add(F.lit("2023-01-01").cast("date"), (F.rand(6) * 731).cast("int")).alias("sale_date"),
        F.element_at(F.array(F.lit("web"), F.lit("app"), F.lit("store")), (F.rand(7) * 3).cast("int") + 1).alias("channel"),
    ).write.mode("overwrite").parquet("data/big/sales")
if not os.path.exists("data/big/customers/_SUCCESS"):
    cities = ["Tashkent", "Samarkand", "Bukhara", "Berlin", "Munich", "Madrid", "London", "Paris", "Rome", "Osaka"]
    spark.range(1, 2_000_001, numPartitions=4).select(
        F.col("id").cast("int").alias("customer_id"),
        F.concat(F.lit("Customer "), F.col("id")).alias("name"),
        F.element_at(F.array(*[F.lit(c) for c in cities]), (F.col("id") % 10).cast("int") + 1).alias("city"),
        F.when(F.col("id") % 7 == 0, "business").otherwise("private").alias("segment"),
    ).write.mode("overwrite").parquet("data/big/customers")
if not os.path.exists("data/big/products/_SUCCESS"):
    categories = ["Books", "Electronics", "Kitchen", "Home", "Toys", "Garden", "Sports", "Beauty", "Music", "Games"]
    spark.range(1, 50_001, numPartitions=1).select(
        F.col("id").cast("int").alias("product_id"),
        F.concat(F.lit("Product "), F.col("id")).alias("product_name"),
        F.element_at(F.array(*[F.lit(c) for c in categories]), (F.col("id") % 10).cast("int") + 1).alias("category"),
        F.round(F.rand(8) * 500 + 5, 2).alias("price"),
    ).write.mode("overwrite").parquet("data/big/products")

sales = spark.read.parquet("data/big/sales")
customers_big = spark.read.parquet("data/big/customers")
products_big = spark.read.parquet("data/big/products")
print(f"sales: {sales.count():,} rows | customers_big: {customers_big.count():,} | products_big: {products_big.count():,}")
sales.show(3)

sales: 10,000,000 rows | customers_big: 2,000,000 | products_big: 50,000
+-------+-----------+----------+--------+--------+----------+-------+
|sale_id|customer_id|product_id|store_id|quantity| sale_date|channel|
+-------+-----------+----------+--------+--------+----------+-------+
|2500001|          1|      1196|      15|       2|2023-12-20|  store|
|2500002|    1906291|      4755|     148|       3|2024-06-20|    app|
|2500003|     132273|     38526|      97|       4|2023-07-05|    web|
+-------+-----------+----------+--------+--------+----------+-------+
only showing top 3 rows


Print every table, so you can check results by eye:

In [ ]:
for name in ["customers", "products", "orders", "reviews", "signups_raw", "customers_new", "promotions", "events", "referrals"]:
    print(name)
    globals()[name].show(truncate=False)

customers
+-----------+----------------+----------------+--------+-----------+
|customer_id|name            |email           |city    |signup_date|
+-----------+----------------+----------------+--------+-----------+
|1          |Aziz Karimov    |aziz@mail.uz    |Tashkent|2024-01-05 |
|2          |Bella Schmidt   |bella@web.de    |Berlin  |2024-01-20 |
|3          |Carlos Ruiz     |carlos@correo.es|Madrid  |2024-02-02 |
|4          |Dilnoza Rahimova|dilnoza@mail.uz |Tashkent|2024-02-14 |
|5          |Emma Wilson     |emma@post.co.uk |London  |2024-03-01 |
|6          |Farid Aliyev    |NULL            |NULL    |2024-03-10 |
|7          |Grace Lee       |grace@web.de    |Berlin  |2024-03-15 |
+-----------+----------------+----------------+--------+-----------+

products
+----------+------------+-----------+------+
|product_id|product_name|category   |price |
+----------+------------+-----------+------+
|101       |Laptop      |Electronics|1200.0|
|102       |Headphones  |Electronics|150.

## Lesson 1 · Join types

`left.join(right, on, how)`:

| `how` | Keeps |
|---|---|
| `"inner"` (the default) | the rows that have a match on both sides |
| `"left"` (also `"left_outer"`) | every left row; the right columns are null where nothing matched |
| `"right"` | every right row |
| `"full"` (also `"outer"`, `"full_outer"`) | every row of both sides |
| `"left_semi"` | the left rows that **have** a match: only the left columns, and never duplicated (SQL `WHERE EXISTS`) |
| `"left_anti"` | the left rows that have **no** match: only the left columns (SQL `WHERE NOT EXISTS`) |
| `a.crossJoin(b)` | every combination: rows of a × rows of b (Lesson 4) |

**Three ways to write `on`:**

| `on=` | Result |
|---|---|
| `"customer_id"` | the same name on both sides; the output has **one** customer_id column |
| `["customer_id", "day"]` | several keys with the same names |
| `orders.customer_id == customers.customer_id`, or with different names `mines.id == extraction.mine_id` | an expression; the output keeps **both** key columns (Lesson 3) |

**What decides the number of rows:**
* **A null key never matches**, not even another null. The guest order (no customer_id) drops out of every inner join on customer_id.
* **One-to-many repeats rows**: a customer with 4 orders appears 4 times after the join (and Day 13, Lesson 6: two "many" tables multiply).
* **After a full join** on an expression, the key of the side without a match is null: combine with `F.coalesce(a.key, b.key)`. With `on="key"`, Spark does that for you.

On Zillacode, an "all X, even without Y" sentence means a left join; "only X that have Y" means inner or left_semi; "X that never..." means left_anti; "all records from both" means full (#14, #24, #47). [join docs](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.join.html).

In [ ]:
c = customers.select("customer_id", "name")
o = orders.select("order_id", "customer_id", "status")
for how in ["inner", "left", "right", "full", "left_semi", "left_anti"]:
    print(f"{how:10}", c.join(o, "customer_id", how).count(), "rows")

inner      15 rows
left       16 rows
right      16 rows
full       17 rows
left_semi  6 rows
left_anti  1 rows


In [ ]:
# left: Grace (7) has no orders, so her order columns are null. right: the guest order has no customer.
c.join(o, "customer_id", "left").filter(F.col("customer_id") >= 6).orderBy("customer_id").show()
c.join(o, "customer_id", "right").filter(F.col("customer_id").isNull()).show()

+-----------+------------+--------+-------+
|customer_id|        name|order_id| status|
+-----------+------------+--------+-------+
|          6|Farid Aliyev|    1012|shipped|
|          7|   Grace Lee|    NULL|   NULL|
+-----------+------------+--------+-------+

+-----------+----+--------+---------+
|customer_id|name|order_id|   status|
+-----------+----+--------+---------+
|       NULL|NULL|    1011|delivered|
+-----------+----+--------+---------+



In [ ]:
# semi and anti: only the left columns, at most one row per left row
c.join(o, "customer_id", "left_semi").orderBy("customer_id").show()
c.join(o, "customer_id", "left_anti").show()

+-----------+----------------+
|customer_id|            name|
+-----------+----------------+
|          1|    Aziz Karimov|
|          2|   Bella Schmidt|
|          3|     Carlos Ruiz|
|          4|Dilnoza Rahimova|
|          5|     Emma Wilson|
|          6|    Farid Aliyev|
+-----------+----------------+

+-----------+---------+
|customer_id|     name|
+-----------+---------+
|          7|Grace Lee|
+-----------+---------+



### Exercises

**Q1 · `etl(customers, orders)`** Which customers have never placed an order? Show customer_id and name.

Expected output:

```
+-----------+---------+
|customer_id|     name|
+-----------+---------+
|          7|Grace Lee|
+-----------+---------+
```

In [ ]:
def etl(customers, orders):
    # Write code here: return a DataFrame
    return customers

etl(customers, orders).show()

**Q2 · `etl(customers, orders)`** Show every customer with their number of orders, including customers with 0 orders. Show name and orders, most orders first; on a tie, by name.

Expected output:

```
+----------------+------+
|            name|orders|
+----------------+------+
|    Aziz Karimov|     4|
|   Bella Schmidt|     3|
|     Carlos Ruiz|     3|
|Dilnoza Rahimova|     2|
|     Emma Wilson|     2|
|    Farid Aliyev|     1|
|       Grace Lee|     0|
+----------------+------+
```

In [ ]:
def etl(customers, orders):
    # Write code here: return a DataFrame
    return customers

etl(customers, orders).show()

**Q3 · `etl(orders, customers)`** Which customers had all of their orders delivered (and at least one order)? Show their names, from A to Z.

Expected output:

```
+-------------+
|         name|
+-------------+
|Bella Schmidt|
+-------------+
```

In [ ]:
def etl(orders, customers):
    # Write code here: return a DataFrame
    return orders

etl(orders, customers).show()

## Lesson 2 · 🔬 How Spark executes a join

Rows that match must meet in the same task. Spark has five algorithms for that (theory Day 5), and each one leaves its own shape in the plan:

| Algorithm | How it works | In the plan | Cost |
|---|---|---|---|
| **Broadcast hash join** | the small side is collected to the driver and a copy goes to every task, which builds a hash table from it; each task then streams its part of the big side through that table | `BroadcastHashJoin`, with a `BroadcastExchange` on the small side and **no `Exchange` on the big side** | the big side isn't moved at all: the cheapest join, as long as the small side fits in memory |
| **Shuffle hash join** | both sides are shuffled by the key; each task builds a hash table from its part of one side | `ShuffledHashJoin`, an `Exchange` on each side, no `Sort` | two shuffles; one side's partition must fit in memory as a hash table |
| **Sort-merge join** | both sides are shuffled by the key and sorted; each task walks the two sorted lists side by side | `SortMergeJoin`, with `Exchange` and `Sort` on each side | two shuffles and two sorts, but sorting can spill to disk, so it rarely runs out of memory: the default for two big tables |
| **Broadcast nested loop join** | one side is broadcast, and every row of the other side is compared with **every** broadcast row | `BroadcastNestedLoopJoin` | rows × rows comparisons: what you get without an `=` in the condition (range joins, cross joins) |
| **Cartesian product** | every partition of one side with every partition of the other | `CartesianProduct` | the same comparisons, without the broadcast: the most expensive |

**How Spark picks one** ([hints in the tuning guide](https://spark.apache.org/docs/latest/sql-performance-tuning.html#join-strategy-hints); for the curious, `JoinSelection` in [SparkStrategies.scala](https://github.com/apache/spark/blob/master/sql/core/src/main/scala/org/apache/spark/sql/execution/SparkStrategies.scala)):
1. A **hint** wins, if that algorithm can run this join: `F.broadcast(df)` (or `df.hint("broadcast")`), `df.hint("merge")`, `df.hint("shuffle_hash")`, `df.hint("shuffle_replicate_nl")`.
2. With an **equality** in the condition: **broadcast** a side whose *estimated* size is under `spark.sql.autoBroadcastJoinThreshold` (10 MB) and which this join type can broadcast (next table); otherwise **sort-merge**. (Without a hint, shuffle hash is chosen only if you turn `spark.sql.join.preferSortMergeJoin` off.)
3. With **no** equality: broadcast nested loop when a side can be broadcast, otherwise a Cartesian product.

**Which side can be broadcast?** Not a side whose every row must appear in the result: a hash table doesn't tell Spark which of its rows never matched.

| Join type | Can broadcast |
|---|---|
| inner | either side |
| `left`, `left_semi`, `left_anti` | the **right** side only |
| `right` | the **left** side only |
| `full` | neither: a full join always shuffles |

So `F.broadcast(customers).join(orders, "customer_id", "left")` ignores the hint (example below).

**Estimates, not facts.** The planner estimates each side from its files: a Parquet table's size on disk, scaled down to the columns the query uses (`explain(mode="cost")` prints the estimates). It doesn't know how many rows a filter keeps, unless you've computed table statistics, so a filtered table looks as big as the whole file. **AQE corrects this at run time**: when the shuffle of a sort-merge join has run, it knows the real size of each side, and if one turned out small, it **switches to a broadcast hash join** ([docs](https://spark.apache.org/docs/latest/sql-performance-tuning.html#converting-sort-merge-join-to-broadcast-join); in the final plan, the big side reads its shuffle files with `AQEShuffleRead local`, without the network). The switch saves the sort and the network, but the big side has already been shuffled to disk by then. When you know a side is small (a filtered or aggregated dimension), `F.broadcast(...)` saves that shuffle as well.

In [ ]:
# Small tables: products (under 10 MB) is broadcast, orders is not moved
orders.join(products, "product_id").explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [product_id#57, order_id#55, customer_id#56, quantity#58, order_date#59, status#60, product_name#52, category#53, price#54]
   +- BroadcastHashJoin [product_id#57], [product_id#51], Inner, BuildRight, false, false
      :- Filter isnotnull(product_id#57)
      :  +- FileScan parquet [order_id#55,customer_id#56,product_id#57,quantity#58,order_date#59,status#60] Batched: true, DataFilters: [isnotnull(product_id#57)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/you/spark-lab/data/shop/orders], PartitionFilters: [], PushedFilters: [IsNotNull(product_id)], ReadSchema: struct<order_id:int,customer_id:int,product_id:int,quantity:int,order_date:date,status:string>
      +- BroadcastExchange HashedRelationBroadcastMode(List(cast(input[0, int, false] as bigint)),false), [plan_id=1733]
         +- Filter isnotnull(product_id#51)
            +- FileScan parquet [product_id#51,product_name#52,category#53,price#

In [ ]:
# With broadcasting switched off, the same join becomes a sort-merge join: Exchange + Sort on each side
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
orders.join(products, "product_id").explain()
# The shuffle_hash hint: an Exchange on each side, but no Sort
orders.join(products.hint("shuffle_hash"), "product_id").explain()
spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [product_id#57, order_id#55, customer_id#56, quantity#58, order_date#59, status#60, product_name#52, category#53, price#54]
   +- SortMergeJoin [product_id#57], [product_id#51], Inner
      :- Sort [product_id#57 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(product_id#57, 200), ENSURE_REQUIREMENTS, [plan_id=1763]
      :     +- Filter isnotnull(product_id#57)
      :        +- FileScan parquet [order_id#55,customer_id#56,product_id#57,quantity#58,order_date#59,status#60] Batched: true, DataFilters: [isnotnull(product_id#57)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/you/spark-lab/data/shop/orders], PartitionFilters: [], PushedFilters: [IsNotNull(product_id)], ReadSchema: struct<order_id:int,customer_id:int,product_id:int,quantity:int,order_date:date,status:string>
      +- Sort [product_id#51 ASC NULLS FIRST], false, 0
         +- Exchange hashpartitioning(product_id#51, 200)

In [ ]:
# A left join can only broadcast its right side: the hint on customers is ignored, orders is broadcast instead
F.broadcast(customers).join(orders, "customer_id", "left").explain()
# A full join can't broadcast either side: sort-merge, even for 7 and 16 rows
customers.join(orders, "customer_id", "full").explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [customer_id#46, name#47, email#48, city#49, signup_date#50, order_id#55, product_id#57, quantity#58, order_date#59, status#60]
   +- BroadcastHashJoin [customer_id#46], [customer_id#56], LeftOuter, BuildRight, false, false
      :- FileScan parquet [customer_id#46,name#47,email#48,city#49,signup_date#50] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/you/spark-lab/data/shop/customers], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<customer_id:int,name:string,email:string,city:string,signup_date:date>
      +- BroadcastExchange HashedRelationBroadcastMode(List(cast(input[1, int, false] as bigint)),false), [plan_id=1822]
         +- Filter isnotnull(customer_id#56)
            +- FileScan parquet [order_id#55,customer_id#56,product_id#57,quantity#58,order_date#59,status#60] Batched: true, DataFilters: [isnotnull(customer_id#56)], Format: Parquet, Location: 

In [ ]:
# Big tables: customers_big is about 18 MB, over the 10 MB limit, so Spark plans a sort-merge join
joined = sales.join(customers_big, "customer_id")
joined.explain()
measure(joined, "sort-merge join", stages=True)
# Broadcasting customers_big anyway: the 10 million sales stay where they are
measure(sales.join(F.broadcast(customers_big), "customer_id"), "broadcast join", stages=True)

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [customer_id#110, sale_id#109L, product_id#111, store_id#112, quantity#113, sale_date#114, channel#115, name#117, city#118, segment#119]
   +- SortMergeJoin [customer_id#110], [customer_id#116], Inner
      :- Sort [customer_id#110 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(customer_id#110, 200), ENSURE_REQUIREMENTS, [plan_id=1875]
      :     +- Filter isnotnull(customer_id#110)
      :        +- FileScan parquet [sale_id#109L,customer_id#110,product_id#111,store_id#112,quantity#113,sale_date#114,channel#115] Batched: true, DataFilters: [isnotnull(customer_id#110)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/you/spark-lab/data/big/sales], PartitionFilters: [], PushedFilters: [IsNotNull(customer_id)], ReadSchema: struct<sale_id:bigint,customer_id:int,product_id:int,store_id:int,quantity:int,sale_date:date,channel:string>
      +- Sort [customer_id#116 ASC NULLS FIRST], false,

In [ ]:
# One day of sales: the planner estimates the filtered side as big as the whole table...
last_day = (sales.filter(F.col("sale_date") == "2024-12-31")
                 .join(customers_big, "customer_id")
                 .select("sale_id", "name", "city", "segment"))
last_day.explain(mode="cost")
# ...so it plans a sort-merge join; at run time AQE sees about 14,000 rows and switches to a broadcast join
measure(last_day, "last day x customers", plan=True)

== Optimized Logical Plan ==
Project [sale_id#109L, name#117, city#118, segment#119], Statistics(sizeInBytes=751.4 TiB)
+- Join Inner, (customer_id#110 = customer_id#116), Statistics(sizeInBytes=830.5 TiB)
   :- Project [sale_id#109L, customer_id#110], Statistics(sizeInBytes=47.9 MiB)
   :  +- Filter ((isnotnull(sale_date#114) AND (sale_date#114 = 2024-12-31)) AND isnotnull(customer_id#110)), Statistics(sizeInBytes=134.3 MiB)
   :     +- Relation [sale_id#109L,customer_id#110,product_id#111,store_id#112,quantity#113,sale_date#114,channel#115] parquet, Statistics(sizeInBytes=134.3 MiB)
   +- Filter isnotnull(customer_id#116), Statistics(sizeInBytes=17.3 MiB)
      +- Relation [customer_id#116,name#117,city#118,segment#119] parquet, Statistics(sizeInBytes=17.3 MiB)

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [sale_id#109L, name#117, city#118, segment#119]
   +- SortMergeJoin [customer_id#110], [customer_id#116], Inner
      :- Sort [customer_id#110 ASC NULLS FIRST

### Exercises

**Q4 · 🔍 Read the plan** **Predict** the algorithm of each join, and which side is broadcast (if any). Then run the cell and check.

In [ ]:
# (a)
orders.join(customers, "customer_id", "left").explain()
# (b)
orders.join(customers, "customer_id", "left_anti").explain()
# (c)
customers_new.join(customers, "customer_id", "full").explain()
# (d)
products.join(promotions, products.price > promotions.discount_pct * 10).explain()
# Your answers:

**Q5 · ⚡ `etl(sales, customers_big)`** For every city and segment, count the sales and add up the items, over all 10 million sales. Sort by city, then segment. `slow_etl` gets it right with a sort-merge join; make Spark skip the big shuffle.

Expected output:

```
+---------+--------+-------+--------+
|     city| segment|  sales|   items|
+---------+--------+-------+--------+
|   Berlin|business| 100333|  300332|
|   Berlin| private| 599779| 1797005|
|  Bukhara|business|  99825|  300374|
|  Bukhara| private| 600575| 1802763|
|   London|business| 100257|  301404|
|   London| private| 598971| 1796369|
|   Madrid|business|  99516|  298603|
|   Madrid| private| 600367| 1801061|
|   Munich|business| 100401|  302030|
|   Munich| private| 599394| 1797227|
|    Osaka|business|  99878|  300327|
|    Osaka| private| 600443| 1802890|
|    Paris|business| 100229|  300910|
|    Paris| private| 600763| 1801993|
|     Rome|business| 100076|  300206|
|     Rome| private| 600013| 1800734|
|Samarkand|business| 100030|  299900|
|Samarkand| private|3597732|10793985|
| Tashkent|business| 100215|  300142|
| Tashkent| private| 601203| 1806120|
+---------+--------+-------+--------+
```

**Target:** a `BroadcastHashJoin` with no `Exchange` on the sales side, and a few KB of shuffle in `measure` instead of about 100 MB.

In [ ]:
def slow_etl(sales, customers_big):
    return (sales.join(customers_big, "customer_id")
                 .groupBy("city", "segment")
                 .agg(F.count("*").alias("sales"), F.sum("quantity").alias("items"))
                 .orderBy("city", "segment"))

def etl(sales, customers_big):
    # Write code here: return a DataFrame
    return slow_etl(sales, customers_big)

measure(slow_etl(sales, customers_big), "slow")
measure(etl(sales, customers_big), "yours")
etl(sales, customers_big).show()
etl(sales, customers_big).explain()

## Lesson 3 · Same column names, one table in two roles, self-joins

After a join on an expression, the columns of **both** sides are there. If both sides have a column with the same name (`id`, `name`, `customer_id`), the plain name is **ambiguous**, and Spark stops with `[AMBIGUOUS_REFERENCE]`. Three fixes:

1. **Pick through the DataFrame**: `select(stars["name"].alias("star_name"), planets["name"].alias("planet_name"))`. Zillacode's #33 and #52 do this.
2. **Alias the DataFrames** and use dotted names: `a = df.alias("a")`, then `F.col("a.name")`. #27 and #42 do this. It's the only way in a **self-join**, where both sides are the same DataFrame.
3. **Rename before the join**, so there is nothing to confuse: `airports.withColumnRenamed("airport_name", "origin_name")`. #49 does this. It's the cleanest choice when one table plays **two roles**.

**One table in two roles** (origin and destination airport; the customer who invited and the customer who joined) means two joins with two renamed copies of the same table.

**A self-join** compares rows of one table with each other: pairs in the same city, employees and their managers, an order and the same customer's earlier orders. To get each pair once, and never pair a row with itself, add `a.id < b.id` to the condition.

In [ ]:
# Ambiguous: both sides have customer_id after a join on an expression
try:
    c.join(o, c.customer_id == o.customer_id).select("customer_id").show()
except Exception as e:
    print(type(e).__name__, "->", str(e).split(".")[0])
# Fix 1: say which side you mean
c.join(o, c.customer_id == o.customer_id).select(c["customer_id"], "name", "order_id").orderBy("order_id").show(3)

AnalysisException -> [AMBIGUOUS_REFERENCE] Reference `customer_id` is ambiguous, could be: [`customer_id`, `customer_id`]
+-----------+-------------+--------+
|customer_id|         name|order_id|
+-----------+-------------+--------+
|          1| Aziz Karimov|    1001|
|          1| Aziz Karimov|    1002|
|          2|Bella Schmidt|    1003|
+-----------+-------------+--------+
only showing top 3 rows


In [ ]:
# One table in two roles: rename it once per role, then join twice
airports = spark.createDataFrame([("TAS", "Tashkent"), ("IST", "Istanbul"), ("BER", "Berlin")], "code string, city string")
flights = spark.createDataFrame([(1, "TAS", "IST"), (2, "IST", "BER"), (3, "BER", "TAS")], "flight_id int, origin string, destination string")
origin = airports.select(F.col("code").alias("origin"), F.col("city").alias("origin_city"))
destination = airports.select(F.col("code").alias("destination"), F.col("city").alias("destination_city"))
(flights.join(origin, "origin")
        .join(destination, "destination")
        .select("flight_id", "origin_city", "destination_city")
        .orderBy("flight_id")
        .show())

+---------+-----------+----------------+
|flight_id|origin_city|destination_city|
+---------+-----------+----------------+
|        1|   Tashkent|        Istanbul|
|        2|   Istanbul|          Berlin|
|        3|     Berlin|        Tashkent|
+---------+-----------+----------------+



In [ ]:
# Self-join: pairs of products in the same category, each pair once
a, b = products.alias("a"), products.alias("b")
(a.join(b, (F.col("a.category") == F.col("b.category")) & (F.col("a.product_id") < F.col("b.product_id")))
  .select(F.col("a.category").alias("category"),
          F.col("a.product_name").alias("product_a"),
          F.col("b.product_name").alias("product_b"))
  .orderBy("category")
  .show())

+-----------+------------+----------+
|   category|   product_a| product_b|
+-----------+------------+----------+
|      Books|       Novel|  Cookbook|
|Electronics|      Laptop|Headphones|
|    Kitchen|Coffee Maker|   Blender|
+-----------+------------+----------+



### Exercises

**Q6 · `etl(referrals, customers)`** Show every referral with both names: referral_date, referrer (the name of the customer who invited) and referred (the name of the customer who joined; empty if unknown). Sort by referral_date.

Expected output:

```
+-------------+----------------+----------------+
|referral_date|        referrer|        referred|
+-------------+----------------+----------------+
|   2024-02-10|    Aziz Karimov|Dilnoza Rahimova|
|   2024-02-25|Dilnoza Rahimova|     Emma Wilson|
|   2024-03-08|    Aziz Karimov|    Farid Aliyev|
|   2024-03-12|   Bella Schmidt|       Grace Lee|
|   2024-03-30|   Bella Schmidt|            NULL|
+-------------+----------------+----------------+
```

In [ ]:
def etl(referrals, customers):
    # Write code here: return a DataFrame
    return referrals

etl(referrals, customers).show()

**Q7 · `etl(customers)`** Show the pairs of customers who live in the same city: city, customer_a and customer_b (names). Each pair once, customer_a being the one with the smaller customer_id. Sort by city.

Expected output:

```
+--------+-------------+----------------+
|    city|   customer_a|      customer_b|
+--------+-------------+----------------+
|  Berlin|Bella Schmidt|       Grace Lee|
|Tashkent| Aziz Karimov|Dilnoza Rahimova|
+--------+-------------+----------------+
```

In [ ]:
def etl(customers):
    # Write code here: return a DataFrame
    return customers

etl(customers).show()

## Lesson 4 · Joining on a range, and cross joins

A join condition can be any true/false expression, not only `==`:
* **A date inside a period**: `F.col("order_date").between(F.col("start_date"), F.col("end_date"))`, usually together with an equality (`products.category == promotions.category`). Zillacode's #22 (minutes watched while a subscription was active) is this pattern.
* **A value inside a band**: price bands, tax brackets, age groups.

A join on equality plus a range is fast. A join on a range alone compares every pair of rows, which is fine for small tables like Zillacode's.

`crossJoin` pairs every row of one side with every row of the other (Zillacode #38). Its typical job is to **build a complete grid** (every customer × every category, every day × every store), then `left` join the real numbers onto it and fill the gaps with 0. That way, combinations without any data still appear in the answer.

🔬 **In the plan.** A condition without `=` leaves Spark only the nested-loop algorithms (Lesson 2): `BroadcastNestedLoopJoin` when a side can be broadcast, `CartesianProduct` otherwise. Both compare every pair of rows: 10 million sales × 1,000 promotions would be 10 billion comparisons. **Add an equality whenever the data allows it**, like `category` in the next cell: Spark then runs a hash or sort-merge join on the equal part and checks the range only for the pairs that already match (the range appears as the join's extra condition).

In [ ]:
bands = spark.createDataFrame([("budget", 0.0, 49.99), ("standard", 50.0, 199.99), ("premium", 200.0, 99999.0)],
                              "band string, low double, high double")
(products.join(bands, F.col("price").between(F.col("low"), F.col("high")))
         .select("product_name", "price", "band")
         .orderBy("price")
         .show())

+------------+------+--------+
|product_name| price|    band|
+------------+------+--------+
|       Novel|  15.0|  budget|
|    Cookbook|  25.0|  budget|
|   Desk Lamp|  40.0|  budget|
|     Blender|  60.0|standard|
|Coffee Maker|  85.0|standard|
|  Headphones| 150.0|standard|
|      Laptop|1200.0| premium|
+------------+------+--------+



In [ ]:
# A full grid of month × status, then the real counts, then 0 for the gaps
months = orders.select(F.month("order_date").alias("month")).distinct()
statuses = orders.select("status").distinct()
counts = orders.groupBy(F.month("order_date").alias("month"), "status").agg(F.count("*").alias("orders"))
(months.crossJoin(statuses)
       .join(counts, ["month", "status"], "left")
       .fillna(0)
       .orderBy("month", "status")
       .show())

+-----+---------+------+
|month|   status|orders|
+-----+---------+------+
|    1|cancelled|     0|
|    1|delivered|     3|
|    1|  shipped|     0|
|    2|cancelled|     1|
|    2|delivered|     2|
|    2|  shipped|     1|
|    3|cancelled|     1|
|    3|delivered|     5|
|    3|  shipped|     3|
+-----+---------+------+



In [ ]:
# Range only: a nested loop, every order compared with every promotion
orders.join(promotions, orders.order_date.between(promotions.start_date, promotions.end_date)).explain()
# Equality + range: a hash join on category; the dates are checked only for pairs with the same category
with_category = orders.join(products, "product_id")
cond = (with_category.category == promotions.category) & with_category.order_date.between(promotions.start_date, promotions.end_date)
with_category.join(promotions, cond).explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastNestedLoopJoin BuildRight, Inner, ((order_date#59 >= start_date#82) AND (order_date#59 <= end_date#83))
   :- Filter isnotnull(order_date#59)
   :  +- FileScan parquet [order_id#55,customer_id#56,product_id#57,quantity#58,order_date#59,status#60] Batched: true, DataFilters: [isnotnull(order_date#59)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/you/spark-lab/data/shop/orders], PartitionFilters: [], PushedFilters: [IsNotNull(order_date)], ReadSchema: struct<order_id:int,customer_id:int,product_id:int,quantity:int,order_date:date,status:string>
   +- BroadcastExchange IdentityBroadcastMode, [plan_id=4153]
      +- Filter (isnotnull(start_date#82) AND isnotnull(end_date#83))
         +- FileScan parquet [promo_id#80,category#81,start_date#82,end_date#83,discount_pct#84] Batched: true, DataFilters: [isnotnull(start_date#82), isnotnull(end_date#83)], Format: Parquet, Location: InMemoryFileIndex(1 paths

### Exercise

**Q8 · `etl(orders, products, promotions)`** For each delivered order, find the promotion of its product's category that was running on the order date. Show order_id, product_name, promo_id and discount_pct; orders without a promotion drop out. Sort by order_id.

Expected output:

```
+--------+------------+--------+------------+
|order_id|product_name|promo_id|discount_pct|
+--------+------------+--------+------------+
|    1001|      Laptop|  WINTER|          10|
|    1003|  Headphones|  WINTER|          10|
|    1009|       Novel|BOOKWEEK|          20|
|    1011|     Blender|  SPRING|          15|
|    1013|      Laptop|   FLASH|           5|
+--------+------------+--------+------------+
```

In [ ]:
def etl(orders, products, promotions):
    # Write code here: return a DataFrame
    return orders

etl(orders, products, promotions).show()

## Lesson 5 · 🔬 Skew: when one key has too many rows

In `sales`, customer 1 (the reseller) has 3 million of the 10 million rows. A shuffle sends all rows of a key to the **same** partition, so one task gets 3 million rows while every other task gets about 35,000. A stage lasts as long as its slowest task, and that task may also run short of memory and spill to disk (theory Day 6).

**How to see skew**
* In the data: count the rows per key (`groupBy(key).count()`, most first), or the rows per partition after a shuffle with `F.spark_partition_id()` (first cell below).
* In the Spark UI: open the join's stage in the *Stages* tab. In *Summary Metrics*, the **Max** duration and shuffle read are far above the **Median**. `measure(..., stages=True)` prints the median and the slowest task of every stage.

**What suffers from a hot key**
* **Joins** that shuffle (sort-merge, shuffle hash): all rows of the key meet in one task.
* **Windows** partitioned by the key: one task sorts and scans the whole partition, and AQE can't split it (Q9).
* **Not `groupBy` with `sum`, `count`, `avg`...**: partial aggregation (Day 13, Lesson 5) sends one row per key per task, so the hot key costs almost nothing extra. (`collect_list` of the hot key does build one huge list.)

**Fixes for a skewed join, best first**
1. **Broadcast the other side** when it's small enough: no shuffle, so no skew (Lesson 2).
2. **AQE's skew join** (on by default, [docs](https://spark.apache.org/docs/latest/sql-performance-tuning.html#optimizing-skew-join)): after the shuffle, AQE looks for partitions bigger than 5 × the median (`spark.sql.adaptive.skewJoin.skewedPartitionFactor`) **and** bigger than 256 MB (`skewedPartitionThresholdInBytes`). It splits such a partition into pieces of about 64 MB (`advisoryPartitionSizeInBytes`) and joins each piece with a copy of the other side's matching partition. The final plan shows `SortMergeJoin(skew=true)` and `AQEShuffleRead skewed`. It works for sort-merge and shuffle hash joins (for left and right joins, on the side whose rows are all kept). Two things to know: on a laptop our hot partition is under 256 MB, so the examples lower the thresholds; and AQE measures partitions in **compressed shuffle bytes**, not rows. Customer 1's partition is about 70 MB when every column is shuffled, but only about 10 MB when just `customer_id` and `quantity` are: then AQE sees no skew, while that task still processes 3 million rows.
3. **Salting**, when AQE can't help (an older Spark, a full join, a hot key that compresses well): give the big side's rows of the hot key a random number 0..N-1 (the *salt*), copy the other side's row for that key N times (one per salt), and join on (key, salt). The hot key's rows now spread over N partitions. Salting only the hot keys keeps the copies small; salting every key copies the whole other side N times.
4. **Split the work**: join the hot key's rows with a broadcast join, the rest with a normal join, and `union` the two results.

In [ ]:
# Rows per shuffle partition when the sales are hash-partitioned by customer_id (200 partitions)
per_partition = (sales.repartition(200, "customer_id")
                      .groupBy(F.spark_partition_id().alias("partition")).count())
per_partition.agg(F.min("count").alias("min_rows"),
                  F.percentile("count", 0.5).alias("median_rows"),
                  F.max("count").alias("max_rows")).show()

+--------+-----------+--------+
|min_rows|median_rows|max_rows|
+--------+-----------+--------+
|   33885|    35008.5| 3032973|
+--------+-----------+--------+



In [ ]:
# The sort-merge join with AQE's skew join switched off. Coalescing is off too, so each partition is one task
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "false")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "false")
measure(sales.join(customers_big, "customer_id"), "skewed join", stages=True)
# AQE's skew join on, with laptop-sized thresholds (the defaults are 256 MB and 64 MB)
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "16MB")
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "16MB")
measure(sales.join(customers_big, "customer_id"), "AQE skew join", stages=True, plan=True)
for key in ["spark.sql.adaptive.coalescePartitions.enabled", "spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes",
            "spark.sql.adaptive.advisoryPartitionSizeInBytes"]:
    spark.conf.unset(key)                        # back to the defaults

skewed join: 2.96 s | 3 jobs, 3 stages, 208 tasks | read 160.4 MB | shuffle 305.7 MB | spill 48.3 MB
   stage 176:   4 tasks, median 0.91 s, max 0.93 s | read 142.1 MB | shuffle read 0 B, write 271.5 MB
   stage 177:   4 tasks, median 0.33 s, max 0.33 s | read 18.4 MB | shuffle read 0 B, write 34.3 MB
   stage 180: 200 tasks, median 0.01 s, max 1.28 s | read 0 B | shuffle read 305.7 MB, write 0 B
AQE skew join: 2.34 s | 3 jobs, 3 stages, 211 tasks | read 160.4 MB | shuffle 305.7 MB | spill 0 B
   stage 181:   4 tasks, median 0.94 s, max 0.95 s | read 142.1 MB | shuffle read 0 B, write 271.5 MB
   stage 182:   4 tasks, median 0.21 s, max 0.22 s | read 18.4 MB | shuffle read 0 B, write 34.3 MB
   stage 185: 203 tasks, median 0.01 s, max 0.28 s | read 0 B | shuffle read 306.2 MB, write 0 B
== Physical Plan ==
OverwriteByExpression noop-table (27)
+- AdaptiveSparkPlan (26)
   +- == Final Plan ==
      ResultQueryStage (17)
      +- * Project (16)
         +- * SortMergeJoin(skew=true) Inne

In [ ]:
# A groupBy on the hot key is not a problem: partial aggregation sends one row per key from each task
measure(sales.groupBy("customer_id").agg(F.sum("quantity").alias("items")), "groupBy customer_id", stages=True)

groupBy customer_id: 1.18 s | 2 jobs, 2 stages, 8 tasks | read 36.5 MB | shuffle 46.9 MB | spill 0 B
   stage 186:   4 tasks, median 0.83 s, max 0.83 s | read 36.5 MB | shuffle read 0 B, write 46.9 MB
   stage 188:   4 tasks, median 0.31 s, max 0.31 s | read 0 B | shuffle read 46.9 MB, write 0 B


### Exercises

**Q9 · 🔍 Read the plan** This window numbers each customer's sales by date. Run it. Which stage is skewed, and by how much (median vs slowest task)? Why doesn't AQE fix it, and what would you do if you had to run it every day?

In [ ]:
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "false")   # one task per partition
nth = W.partitionBy("customer_id").orderBy("sale_date", "sale_id")
measure(sales.withColumn("nth_purchase", F.row_number().over(nth)), "window by customer", stages=True)
spark.conf.unset("spark.sql.adaptive.coalescePartitions.enabled")
# Your answer:

**Q10 · ⚡ `etl(sales, customers_big)`** Add up the items of each city, sorted by city. In this cell Spark can't broadcast and AQE's skew join is off, so `slow_etl` runs a sort-merge join where one task gets the reseller's 3 million rows. Balance it with salting: spread customer 1's sales over 16 salt values.

Expected output:

```
+---------+--------+
|     city|   items|
+---------+--------+
|   Berlin| 2097337|
|  Bukhara| 2103137|
|   London| 2097773|
|   Madrid| 2099664|
|   Munich| 2099257|
|    Osaka| 2103217|
|    Paris| 2102903|
|     Rome| 2100940|
|Samarkand|11093885|
| Tashkent| 2106262|
+---------+--------+
```

**Target:** in the join stage (200 tasks, about 95 MB of shuffle read), the slowest task (`max`) is at least 2 times faster than with `slow_etl` (3 to 8 times is typical).

In [ ]:
# Two big tables: broadcasting and AQE's skew join are switched off for this exercise
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "false")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "false")

def slow_etl(sales, customers_big):
    return (sales.join(customers_big, "customer_id")
                 .groupBy("city").agg(F.sum("quantity").alias("items"))
                 .orderBy("city"))

def etl(sales, customers_big):
    # Write code here: return a DataFrame
    return slow_etl(sales, customers_big)

try:
    measure(slow_etl(sales, customers_big), "slow", stages=True)
    measure(etl(sales, customers_big), "yours", stages=True)
    etl(sales, customers_big).show()
finally:                                         # back to the defaults, even if etl fails
    for key in ["spark.sql.autoBroadcastJoinThreshold", "spark.sql.adaptive.skewJoin.enabled",
                "spark.sql.adaptive.coalescePartitions.enabled"]:
        spark.conf.unset(key)

## Lesson 6 · Window functions: a group's value on every row

`groupBy` turns each group into **one** row. A **window function** also computes over a group of rows, but it **keeps every row** and writes the result next to it:

```python
w = W.partitionBy("category")                       # the group: rows with the same category
products.withColumn("category_avg", F.avg("price").over(w))
```

* `W.partitionBy(cols)` decides which rows belong together, like `groupBy`. With **no** `partitionBy`, the whole table is one window. That's fine for small data; Spark warns that it moves every row into one partition.
* `.orderBy(cols)` sets the order inside each window. Ranks, `lag`/`lead` and running totals need it (Lessons 7 and 8).
* **Any aggregate** works with `.over(w)`: `sum`, `avg`, `min`, `max`, `count`, `variance` (Zillacode #29)...

Typical uses without an order: compare each row with its group (price minus the category average), a share of the group (`F.col("x") / F.sum("x").over(w)`), the group's size next to each row, and keeping the rows that equal the group's maximum (ties all stay). [Window docs](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/window.html).

🔬 **How a window runs.** A window needs every row of a partition key in one task, sorted: the plan is `Exchange hashpartitioning(partition key)`, then `Sort` (by the key and the window's `orderBy`), then `Window`. Unlike a `groupBy`, there's **no partial aggregation**: every row is shuffled, not one per key. Several windows with the **same `partitionBy`** share one exchange (a different `orderBy` adds a `Sort`). A window **without `partitionBy`** becomes `Exchange SinglePartition`: the whole table in one task, fine on Zillacode, a disaster on 10 million rows.

A group's total next to every row can also be written as `groupBy` + join back. Which is cheaper? The window shuffles every row once. The `groupBy` shuffles almost nothing (partial aggregation), and its small result can be broadcast back, so the big table isn't shuffled at all, but only if you say so: the planner can't know that an aggregate is small, so without `F.broadcast` it plans a sort-merge join and shuffles the big table anyway. The cells below measure all three.

In [ ]:
w = W.partitionBy("category")
products.select(
    "product_name", "category", "price",
    F.round(F.avg("price").over(w), 2).alias("category_avg"),
    F.max("price").over(w).alias("category_max"),
    F.count("*").over(w).alias("in_category"),
    F.round(F.col("price") / F.sum("price").over(w) * 100, 1).alias("pct_of_category"),
).orderBy("category", "price").show()

+------------+-----------+------+------------+------------+-----------+---------------+
|product_name|   category| price|category_avg|category_max|in_category|pct_of_category|
+------------+-----------+------+------------+------------+-----------+---------------+
|       Novel|      Books|  15.0|        20.0|        25.0|          2|           37.5|
|    Cookbook|      Books|  25.0|        20.0|        25.0|          2|           62.5|
|  Headphones|Electronics| 150.0|       675.0|      1200.0|          2|           11.1|
|      Laptop|Electronics|1200.0|       675.0|      1200.0|          2|           88.9|
|   Desk Lamp|       Home|  40.0|        40.0|        40.0|          1|          100.0|
|     Blender|    Kitchen|  60.0|        72.5|        85.0|          2|           41.4|
|Coffee Maker|    Kitchen|  85.0|        72.5|        85.0|          2|           58.6|
+------------+-----------+------+------------+------------+-----------+---------------+



In [ ]:
# Keep the rows that equal their group's maximum (ties all stay)
(orders.filter(F.col("customer_id").isNotNull())
       .withColumn("max_qty", F.max("quantity").over(W.partitionBy("customer_id")))
       .filter(F.col("quantity") == F.col("max_qty"))
       .select("customer_id", "order_id", "quantity")
       .orderBy("customer_id", "order_id")
       .show())

+-----------+--------+--------+
|customer_id|order_id|quantity|
+-----------+--------+--------+
|          1|    1002|       3|
|          2|    1003|       2|
|          3|    1009|       4|
|          4|    1006|       2|
|          4|    1010|       2|
|          5|    1014|       5|
|          6|    1012|       2|
+-----------+--------+--------+



In [ ]:
# Three windows, one partitionBy: one Exchange, two Sorts (one per orderBy), three Window operators
w_date = W.partitionBy("customer_id").orderBy("order_date")
w_qty = W.partitionBy("customer_id").orderBy(F.desc("quantity"))
(orders.withColumn("nth_order", F.row_number().over(w_date))
       .withColumn("qty_rank", F.rank().over(w_qty))
       .withColumn("customer_items", F.sum("quantity").over(W.partitionBy("customer_id")))
       .explain())

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Window [sum(quantity#58) windowspecdefinition(customer_id#56, specifiedwindowframe(RowFrame, unboundedpreceding$(), unboundedfollowing$())) AS customer_items#905L], [customer_id#56]
   +- Window [rank(quantity#58) windowspecdefinition(customer_id#56, quantity#58 DESC NULLS LAST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS qty_rank#896], [customer_id#56], [quantity#58 DESC NULLS LAST]
      +- Sort [customer_id#56 ASC NULLS FIRST, quantity#58 DESC NULLS LAST], false, 0
         +- Window [row_number() windowspecdefinition(customer_id#56, order_date#59 ASC NULLS FIRST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS nth_order#894], [customer_id#56], [order_date#59 ASC NULLS FIRST]
            +- Sort [customer_id#56 ASC NULLS FIRST, order_date#59 ASC NULLS FIRST], false, 0
               +- Exchange hashpartitioning(customer_id#56, 200), ENSURE_REQUIREMENTS, [plan_id=5655]
   

In [ ]:
# Each sale's store total, three ways
store_totals = sales.groupBy("store_id").agg(F.sum("quantity").alias("store_items"))
measure(sales.withColumn("store_items", F.sum("quantity").over(W.partitionBy("store_id"))), "window")
measure(sales.join(store_totals, "store_id"), "groupBy + join back")
measure(sales.join(F.broadcast(store_totals), "store_id"), "groupBy + broadcast join back")

window: 2.42 s | 2 jobs, 2 stages, 9 tasks | read 141.9 MB | shuffle 268.3 MB | spill 223.2 MB
groupBy + join back: 1.49 s | 4 jobs, 4 stages, 13 tasks | read 156.5 MB | shuffle 268.4 MB | spill 0 B
groupBy + broadcast join back: 0.59 s | 3 jobs, 3 stages, 9 tasks | read 156.5 MB | shuffle 36.5 KB | spill 0 B


In [ ]:
# No partitionBy: all 10 million rows go through one task (Exchange SinglePartition)
measure(sales.withColumn("row_no", F.row_number().over(W.orderBy("sale_id"))), "window without partitionBy", stages=True)

window without partitionBy: 5.10 s | 2 jobs, 2 stages, 5 tasks | read 141.9 MB | shuffle 273.5 MB | spill 288.1 MB
   stage 256:   4 tasks, median 0.70 s, max 0.71 s | read 141.9 MB | shuffle read 0 B, write 273.5 MB
   stage 258:   1 tasks, median 4.36 s, max 4.36 s | read 0 B | shuffle read 273.5 MB, write 0 B


### Exercises

**Q11 · `etl(products)`** Show product_name, category, price and diff_from_avg (the price minus its category's average price, 2 decimals), sorted by category, then price.

Expected output:

```
+------------+-----------+------+-------------+
|product_name|   category| price|diff_from_avg|
+------------+-----------+------+-------------+
|       Novel|      Books|  15.0|         -5.0|
|    Cookbook|      Books|  25.0|          5.0|
|  Headphones|Electronics| 150.0|       -525.0|
|      Laptop|Electronics|1200.0|        525.0|
|   Desk Lamp|       Home|  40.0|          0.0|
|     Blender|    Kitchen|  60.0|        -12.5|
|Coffee Maker|    Kitchen|  85.0|         12.5|
+------------+-----------+------+-------------+
```

In [ ]:
def etl(products):
    # Write code here: return a DataFrame
    return products

etl(products).show()

**Q12 · `etl(orders)`** For the orders of customers 1 and 2, show order_id, customer_id, quantity and pct_of_customer: the order's share of all items that customer ordered, in percent, 1 decimal. Sort by order_id.

Expected output:

```
+--------+-----------+--------+---------------+
|order_id|customer_id|quantity|pct_of_customer|
+--------+-----------+--------+---------------+
|    1001|          1|       1|           16.7|
|    1002|          1|       3|           50.0|
|    1003|          2|       2|           50.0|
|    1005|          2|       1|           25.0|
|    1007|          1|       1|           16.7|
|    1013|          2|       1|           25.0|
|    1015|          1|       1|           16.7|
+--------+-----------+--------+---------------+
```

In [ ]:
def etl(orders):
    # Write code here: return a DataFrame
    return orders

etl(orders).show()

## Lesson 7 · Ranking: `row_number`, `rank`, `dense_rank`

With an `orderBy` in the window, you can number the rows of each partition:

| quantity | `row_number` | `rank` | `dense_rank` |
|---|---|---|---|
| 5 | 1 | 1 | 1 |
| 3 | 2 | 2 | 2 |
| 3 | 3 | 2 | 2 |
| 1 | 4 | 4 | 3 |

* `F.row_number()`: 1, 2, 3, ..., always unique. Ties get an **arbitrary** order unless you add a tie-breaker column to `orderBy`.
* `F.rank()`: ties share a number, and the next numbers are skipped (Olympic ranking).
* `F.dense_rank()`: ties share a number, nothing is skipped. "Without gaps in the ranking" in a Zillacode text means `dense_rank` (#7).
* `F.ntile(4)` splits each partition into 4 buckets; `F.percent_rank()` goes from 0 to 1.

**Recipes**
* **Top N per group**: rank over `W.partitionBy(group).orderBy(F.desc(value))`, then keep `<= N`. If ties should all stay, use `rank` or `dense_rank`; for exactly N rows, use `row_number` with a tie-breaker.
* **The latest (or first) row per key**, and **removing duplicates while choosing which row stays**: `row_number` over `partitionBy(key).orderBy(F.desc(time))`, then keep `== 1` (Zillacode #13, #30, #37).
* **Numbering the whole table**: `W.orderBy(...)` with no partition (#28, #43).

Always **drop the helper column** (`rn`, `rank`) before returning, unless the output schema asks for it: an extra column fails the checker.

🔬 **Top N runs cheaper than it looks.** When a `row_number`, `rank` or `dense_rank` is followed directly by a filter like `<= N` (Spark 3.5 and later), the plan gets a `WindowGroupLimit`: each task keeps only the top N rows of every key **before** the shuffle (`Partial`), then again after it (`Final`), so only a few rows per key travel. The next cell shows it on the big table.

In [ ]:
# Ties: orders 1006 and 1010 both have quantity 2, and five orders have quantity 1
w = W.orderBy(F.desc("quantity"))
w_unique = W.orderBy(F.desc("quantity"), "order_id")     # a tie-breaker makes row_number repeatable
(orders.filter(F.col("customer_id").isin(1, 3, 4))
       .select("order_id", "quantity",
               F.row_number().over(w_unique).alias("row_number"),
               F.rank().over(w).alias("rank"),
               F.dense_rank().over(w).alias("dense_rank"))
       .orderBy("row_number")
       .show())

+--------+--------+----------+----+----------+
|order_id|quantity|row_number|rank|dense_rank|
+--------+--------+----------+----+----------+
|    1009|       4|         1|   1|         1|
|    1002|       3|         2|   2|         2|
|    1006|       2|         3|   3|         3|
|    1010|       2|         4|   3|         3|
|    1001|       1|         5|   5|         4|
|    1004|       1|         6|   5|         4|
|    1007|       1|         7|   5|         4|
|    1015|       1|         8|   5|         4|
|    1016|       1|         9|   5|         4|
+--------+--------+----------+----+----------+



In [ ]:
# The FIRST order of each customer: row_number per customer, oldest first, keep 1, drop the helper
w = W.partitionBy("customer_id").orderBy("order_date", "order_id")
(orders.filter(F.col("customer_id").isNotNull())
       .withColumn("rn", F.row_number().over(w))
       .filter(F.col("rn") == 1)
       .drop("rn")
       .orderBy("customer_id")
       .show())

+--------+-----------+----------+--------+----------+---------+
|order_id|customer_id|product_id|quantity|order_date|   status|
+--------+-----------+----------+--------+----------+---------+
|    1001|          1|       101|       1|2024-01-10|delivered|
|    1003|          2|       102|       2|2024-01-25|delivered|
|    1004|          3|       103|       1|2024-02-03|cancelled|
|    1006|          4|       104|       2|2024-02-15|  shipped|
|    1008|          5|       101|       1|2024-03-02|  shipped|
|    1012|          6|       103|       2|2024-03-12|  shipped|
+--------+-----------+----------+--------+----------+---------+



In [ ]:
# Top 3 sales of each store: WindowGroupLimit (Partial, then Final) around the shuffle
w = W.partitionBy("store_id").orderBy(F.desc("quantity"), "sale_id")
top3 = sales.withColumn("rn", F.row_number().over(w)).filter(F.col("rn") <= 3)
top3.explain()
measure(top3, "top 3 per store")
measure(sales.withColumn("rn", F.row_number().over(w)), "rank every sale")

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Filter (rn#1042 <= 3)
   +- Window [row_number() windowspecdefinition(store_id#112, quantity#113 DESC NULLS LAST, sale_id#109L ASC NULLS FIRST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS rn#1042], [store_id#112], [quantity#113 DESC NULLS LAST, sale_id#109L ASC NULLS FIRST]
      +- WindowGroupLimit [store_id#112], [quantity#113 DESC NULLS LAST, sale_id#109L ASC NULLS FIRST], row_number(), 3, Final
         +- Sort [store_id#112 ASC NULLS FIRST, quantity#113 DESC NULLS LAST, sale_id#109L ASC NULLS FIRST], false, 0
            +- Exchange hashpartitioning(store_id#112, 200), ENSURE_REQUIREMENTS, [plan_id=6318]
               +- WindowGroupLimit [store_id#112], [quantity#113 DESC NULLS LAST, sale_id#109L ASC NULLS FIRST], row_number(), 3, Partial
                  +- Sort [store_id#112 ASC NULLS FIRST, quantity#113 DESC NULLS LAST, sale_id#109L ASC NULLS FIRST], false, 0
                     +- FileSc

### Exercises

**Q13 · `etl(products)`** Show the most expensive product of each category: category, product_name and price, sorted by category.

Expected output:

```
+-----------+------------+------+
|   category|product_name| price|
+-----------+------------+------+
|      Books|    Cookbook|  25.0|
|Electronics|      Laptop|1200.0|
|       Home|   Desk Lamp|  40.0|
|    Kitchen|Coffee Maker|  85.0|
+-----------+------------+------+
```

In [ ]:
def etl(products):
    # Write code here: return a DataFrame
    return products

etl(products).show()

**Q14 · `etl(orders, products)`** Show the top 2 products of each category by items delivered (on a tie, the smaller product_id first). Columns: category, product_name, items and rank; sorted by category, then rank.

Expected output:

```
+-----------+------------+-----+----+
|   category|product_name|items|rank|
+-----------+------------+-----+----+
|      Books|       Novel|   12|   1|
|      Books|    Cookbook|    1|   2|
|Electronics|  Headphones|    3|   1|
|Electronics|      Laptop|    2|   2|
|    Kitchen|     Blender|    1|   1|
+-----------+------------+-----+----+
```

In [ ]:
def etl(orders, products):
    # Write code here: return a DataFrame
    return orders

etl(orders, products).show()

## Lesson 8 · Previous and next rows, running totals

| Expression | Gives |
|---|---|
| `F.lag("x").over(w)` | x of the **previous** row in the window, null for the first row. `F.lag("x", 2)` goes two rows back; `F.lag("x", 1, 0)` gives 0 instead of null. |
| `F.lead("x").over(w)` | x of the **next** row |
| `F.sum("x").over(w.rowsBetween(W.unboundedPreceding, W.currentRow))` | a running total |
| `F.avg("x").over(w.rowsBetween(-2, W.currentRow))` | a moving average of this row and the 2 before it |
| `F.first("x").over(w)`, `F.last("x", ignorenulls=True).over(w)` | the first / last value in the frame |

Typical uses: the change since the previous row (`x - lag(x)`), growth in percent (`(x - lag(x)) / lag(x) * 100`, Zillacode #45), days since the previous order, the previous product of the same customer (#21), detecting a change (`status != lag(status)`), running balances.

**The frame trap.** Once a window has an `orderBy`, an aggregate over it uses a default **frame**: from the first row up to the current row, **plus every row with the same order value** (`rangeBetween(W.unboundedPreceding, W.currentRow)`). So two orders on the same date both get a running total that already includes each other. For a true row-by-row running total, use `rowsBetween(W.unboundedPreceding, W.currentRow)` and a tie-breaker in `orderBy`. For the same reason, `F.last("x").over(w)` with an `orderBy` returns the **current** row's value, because the frame ends there; use `rowsBetween(W.unboundedPreceding, W.unboundedFollowing)` to look at the whole partition. Without an `orderBy`, the frame is always the whole partition (Lesson 6).

🔬 **The frame is in the plan.** `explain()` prints each window's frame: `specifiedwindowframe(RangeFrame, unboundedpreceding$(), currentrow$())` is the default frame of an ordered window, which includes the rows with the same order value; `RowFrame` is what `rowsBetween` gives. When a running total looks wrong, the plan tells you which frame Spark used.

In [ ]:
w = W.partitionBy("customer_id").orderBy("order_date", "order_id")
orders.filter(F.col("customer_id") == 1).select(
    "order_id", "order_date", "quantity",
    F.lag("quantity").over(w).alias("prev_qty"),
    (F.col("quantity") - F.lag("quantity").over(w)).alias("change"),
    F.lead("order_date").over(w).alias("next_date"),
    F.datediff("order_date", F.lag("order_date").over(w)).alias("days_since_prev"),
).orderBy("order_date", "order_id").show()

+--------+----------+--------+--------+------+----------+---------------+
|order_id|order_date|quantity|prev_qty|change| next_date|days_since_prev|
+--------+----------+--------+--------+------+----------+---------------+
|    1001|2024-01-10|       1|    NULL|  NULL|2024-01-10|           NULL|
|    1002|2024-01-10|       3|       1|     2|2024-02-20|              0|
|    1007|2024-02-20|       1|       3|    -2|2024-03-25|             41|
|    1015|2024-03-25|       1|       1|     0|      NULL|             34|
+--------+----------+--------+--------+------+----------+---------------+



In [ ]:
# Orders 1001 and 1002 share a date: the default frame counts them together
w_date = W.partitionBy("customer_id").orderBy("order_date")
w_rows = W.partitionBy("customer_id").orderBy("order_date", "order_id").rowsBetween(W.unboundedPreceding, W.currentRow)
orders.filter(F.col("customer_id") == 1).select(
    "order_id", "order_date", "quantity",
    F.sum("quantity").over(w_date).alias("default_frame"),
    F.sum("quantity").over(w_rows).alias("rows_frame"),
).orderBy("order_date", "order_id").show()

+--------+----------+--------+-------------+----------+
|order_id|order_date|quantity|default_frame|rows_frame|
+--------+----------+--------+-------------+----------+
|    1001|2024-01-10|       1|            4|         1|
|    1002|2024-01-10|       3|            4|         4|
|    1007|2024-02-20|       1|            5|         5|
|    1015|2024-03-25|       1|            6|         6|
+--------+----------+--------+-------------+----------+



In [ ]:
# A moving average, and last() with the default frame vs the whole partition
w = W.partitionBy("customer_id").orderBy("order_date", "order_id")
orders.filter(F.col("customer_id") == 1).select(
    "order_id", "quantity",
    F.round(F.avg("quantity").over(w.rowsBetween(-1, W.currentRow)), 2).alias("avg_of_last_2"),
    F.last("order_id").over(w).alias("last_default_frame"),
    F.last("order_id").over(w.rowsBetween(W.unboundedPreceding, W.unboundedFollowing)).alias("last_of_partition"),
).orderBy("order_id").show()

+--------+--------+-------------+------------------+-----------------+
|order_id|quantity|avg_of_last_2|last_default_frame|last_of_partition|
+--------+--------+-------------+------------------+-----------------+
|    1001|       1|          1.0|              1001|             1015|
|    1002|       3|          2.0|              1002|             1015|
|    1007|       1|          2.0|              1007|             1015|
|    1015|       1|          1.0|              1015|             1015|
+--------+--------+-------------+------------------+-----------------+



In [ ]:
# The default frame (RangeFrame) vs rowsBetween (RowFrame), as Spark prints them
w = W.partitionBy("customer_id").orderBy("order_date")
orders.select("order_id",
              F.sum("quantity").over(w).alias("default_frame"),
              F.sum("quantity").over(w.rowsBetween(W.unboundedPreceding, W.currentRow)).alias("rows_frame")).explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [order_id#55, default_frame#1157L, rows_frame#1158L]
   +- Window [sum(quantity#58) windowspecdefinition(customer_id#56, order_date#59 ASC NULLS FIRST, specifiedwindowframe(RangeFrame, unboundedpreceding$(), currentrow$())) AS default_frame#1157L, sum(quantity#58) windowspecdefinition(customer_id#56, order_date#59 ASC NULLS FIRST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS rows_frame#1158L], [customer_id#56], [order_date#59 ASC NULLS FIRST]
      +- Sort [customer_id#56 ASC NULLS FIRST, order_date#59 ASC NULLS FIRST], false, 0
         +- Exchange hashpartitioning(customer_id#56, 200), ENSURE_REQUIREMENTS, [plan_id=6782]
            +- FileScan parquet [order_id#55,customer_id#56,quantity#58,order_date#59] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/you/spark-lab/data/shop/orders], PartitionFilters: [], PushedFilters: [], ReadSchema: str

### Exercises

**Q15 · `etl(orders)`** For every order of customers 1 and 3, show how many days passed since the same customer's previous order (`days_since_prev`, empty for the first order). Show customer_id, order_id, order_date and days_since_prev, sorted by customer_id, then order_date and order_id.

Expected output:

```
+-----------+--------+----------+---------------+
|customer_id|order_id|order_date|days_since_prev|
+-----------+--------+----------+---------------+
|          1|    1001|2024-01-10|           NULL|
|          1|    1002|2024-01-10|              0|
|          1|    1007|2024-02-20|             41|
|          1|    1015|2024-03-25|             34|
|          3|    1004|2024-02-03|           NULL|
|          3|    1009|2024-03-05|             31|
|          3|    1016|2024-03-28|             23|
+-----------+--------+----------+---------------+
```

In [ ]:
def etl(orders):
    # Write code here: return a DataFrame
    return orders

etl(orders).show()

**Q16 · `etl(orders)`** A running total: for customers 1 and 2, show customer_id, order_id, order_date, quantity and `items_so_far` (the items of this order and all earlier orders of the customer). Sort by customer_id, then order_date and order_id.

Expected output:

```
+-----------+--------+----------+--------+------------+
|customer_id|order_id|order_date|quantity|items_so_far|
+-----------+--------+----------+--------+------------+
|          1|    1001|2024-01-10|       1|           1|
|          1|    1002|2024-01-10|       3|           4|
|          1|    1007|2024-02-20|       1|           5|
|          1|    1015|2024-03-25|       1|           6|
|          2|    1003|2024-01-25|       2|           2|
|          2|    1005|2024-02-05|       1|           3|
|          2|    1013|2024-03-18|       1|           4|
+-----------+--------+----------+--------+------------+
```

In [ ]:
def etl(orders):
    # Write code here: return a DataFrame
    return orders

etl(orders).show()

## Mixed problems
Like Zillacode's Medium and Hard problems: a join, an aggregation and a window together. Write down the steps first. About 15 minutes each.

**Q17 · `etl(orders, customers)`** In each city, find the customer with the most orders (on a tie, the smaller customer_id). Customers without a city are left out. Columns: city, name, orders; sorted by city.

Expected output:

```
+--------+-------------+------+
|    city|         name|orders|
+--------+-------------+------+
|  Berlin|Bella Schmidt|     3|
|  London|  Emma Wilson|     2|
|  Madrid|  Carlos Ruiz|     3|
|Tashkent| Aziz Karimov|     4|
+--------+-------------+------+
```

In [ ]:
def etl(orders, customers):
    # Write code here: return a DataFrame
    return orders

etl(orders, customers).show()

**Q18 · `etl(orders)`** For each customer with at least 2 orders, find the longest gap in days between two consecutive orders. Columns: customer_id, longest_gap_days; sorted by customer_id.

Expected output:

```
+-----------+----------------+
|customer_id|longest_gap_days|
+-----------+----------------+
|          1|              41|
|          2|              42|
|          3|              31|
|          4|              22|
|          5|              18|
+-----------+----------------+
```

In [ ]:
def etl(orders):
    # Write code here: return a DataFrame
    return orders

etl(orders).show()

**Q19 · ⚡ `etl(sales, products_big)`** The top 3 products of each category by items sold: category, rank (1 to 3; on a tie, the smaller product_id first), product_id, product_name and items, sorted by category and rank. Each sale is in `sales` only once (sale_id is unique). `slow_etl` is correct but takes seconds and spills to disk; find its two expensive mistakes in the plan and in `measure`, and fix them.

Expected output:

```
+-----------+----+----------+-------------+-----+
|   category|rank|product_id| product_name|items|
+-----------+----+----------+-------------+-----+
|     Beauty|   1|     41067|Product 41067|  773|
|     Beauty|   2|      5037| Product 5037|  767|
|     Beauty|   3|     25777|Product 25777|  752|
|      Books|   1|      3240| Product 3240|  779|
|      Books|   2|     31590|Product 31590|  754|
|      Books|   3|     44380|Product 44380|  752|
|Electronics|   1|      2961| Product 2961|  771|
|Electronics|   2|     27281|Product 27281|  756|
|Electronics|   3|     44231|Product 44231|  755|
|      Games|   1|     28179|Product 28179|  775|
|      Games|   2|     31579|Product 31579|  770|
|      Games|   3|     36959|Product 36959|  761|
|     Garden|   1|      9465| Product 9465|  778|
|     Garden|   2|     27865|Product 27865|  774|
|     Garden|   3|     36135|Product 36135|  772|
|       Home|   1|     22693|Product 22693|  807|
|       Home|   2|     30933|Product 30933|  783|
|       Home|   3|     47733|Product 47733|  771|
|    Kitchen|   1|     24742|Product 24742|  790|
|    Kitchen|   2|     26492|Product 26492|  767|
|    Kitchen|   3|     48372|Product 48372|  755|
|      Music|   1|     38968|Product 38968|  781|
|      Music|   2|     10298|Product 10298|  780|
|      Music|   3|     31318|Product 31318|  769|
|     Sports|   1|     30516|Product 30516|  768|
|     Sports|   2|     18816|Product 18816|  759|
|     Sports|   3|     21686|Product 21686|  756|
|       Toys|   1|     44304|Product 44304|  786|
|       Toys|   2|     20024|Product 20024|  784|
|       Toys|   3|     24964|Product 24964|  774|
+-----------+----+----------+-------------+-----+
```

**Target:** the same 30 rows, with under 10 MB shuffled and no spill (`slow_etl` shuffles about 500 MB and spills about 1 GB).

In [ ]:
def slow_etl(sales, products_big):
    joined = sales.join(products_big, "product_id").distinct()
    per_product = (joined.groupBy("category", "product_id", "product_name")
                         .agg(F.sum("quantity").alias("items")))
    w = W.partitionBy("category").orderBy(F.desc("items"), "product_id")
    return (per_product.withColumn("rank", F.row_number().over(w))
                       .filter(F.col("rank") <= 3)
                       .select("category", "rank", "product_id", "product_name", "items")
                       .orderBy("category", "rank"))

def etl(sales, products_big):
    # Write code here: return a DataFrame
    return slow_etl(sales, products_big)

measure(slow_etl(sales, products_big), "slow")
measure(etl(sales, products_big), "yours")
etl(sales, products_big).show(30)

## Answers
Try each exercise yourself first. Click an exercise to open its answer.

<details>
<summary><b>Q1</b> &nbsp;Which customers have never placed an order? Show customer_id and name.</summary>

```python
def etl(customers, orders):
    return customers.join(orders, "customer_id", "left_anti").select("customer_id", "name").orderBy("customer_id")

etl(customers, orders).show()
```

</details>

<details>
<summary><b>Q2</b> &nbsp;Show every customer with their number of orders, including customers with 0 orders. Show name and orders, ...</summary>

```python
def etl(customers, orders):
    return (customers.join(orders, "customer_id", "left")
                     .groupBy("customer_id", "name")
                     .agg(F.count("order_id").alias("orders"))
                     .select("name", "orders")
                     .orderBy(F.col("orders").desc(), "name"))

etl(customers, orders).show()
```

`count("order_id")` counts only matched rows, so Grace gets 0; `count("*")` would give her 1.

</details>

<details>
<summary><b>Q3</b> &nbsp;Which customers had all of their orders delivered (and at least one order)? Show their names, from A to Z.</summary>

```python
def etl(orders, customers):
    not_delivered = orders.filter(F.col("status") != "delivered").select("customer_id")
    return (customers.join(orders, "customer_id", "left_semi")
                     .join(not_delivered, "customer_id", "left_anti")
                     .select("name")
                     .orderBy("name"))

etl(orders, customers).show()
```

`left_semi` keeps customers with at least one order (once each), and `left_anti` then removes those with an order that isn't delivered.

</details>

<details>
<summary><b>Q4</b> &nbsp;Predict the algorithm of each join, and which side is broadcast.</summary>

(a) `BroadcastHashJoin ... LeftOuter, BuildRight`: customers, the right side, is broadcast. A left join can only build its right side.

(b) `BroadcastHashJoin ... LeftAnti, BuildRight`: the same rule for `left_anti`. Only `customer_id` is read from customers: an anti join needs nothing else from that side.

(c) `SortMergeJoin ... FullOuter`, with an `Exchange` and a `Sort` on both sides: a full join can't broadcast, not even 7 rows.

(d) `BroadcastNestedLoopJoin BuildLeft`: there's no `=` in the condition, so neither a hash nor a sort-merge join is possible. Spark broadcasts one side (for an inner join it may build either; here products, the side it estimated smaller) and compares every product with every promotion.

</details>

<details>
<summary><b>Q5</b> &nbsp;For every city and segment, count the sales and add up the items, over all 10 million sales. Sort by city ...</summary>

```python
def etl(sales, customers_big):
    return (sales.join(F.broadcast(customers_big), "customer_id")
                 .groupBy("city", "segment")
                 .agg(F.count("*").alias("sales"), F.sum("quantity").alias("items"))
                 .orderBy("city", "segment"))

etl(sales, customers_big).show()
```

The query needs three columns of `customers_big` (`customer_id`, `city`, `segment`), which Spark estimates at about 13 MB: over the 10 MB limit, so it plans a sort-merge join and shuffles every sale, then spills while sorting the reseller's 3 million rows. The hint makes each task build a hash table of the 2 million customers and stream its sales through it; only the small partial aggregates are shuffled. The alternative `spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "50MB")` works too, but it changes every join of the session. Lesson 6 of Day 13 suggests aggregating sales per customer before the join: that helps less here, because 10 million sales still come down to about 1.9 million customers.

</details>

<details>
<summary><b>Q6</b> &nbsp;Show every referral with both names: referral_date, referrer (the name of the customer who invited) and ...</summary>

```python
def etl(referrals, customers):
    referrer = customers.select(F.col("customer_id").alias("referrer_id"), F.col("name").alias("referrer"))
    referred = customers.select(F.col("customer_id").alias("referred_id"), F.col("name").alias("referred"))
    return (referrals.join(referrer, "referrer_id", "left")
                     .join(referred, "referred_id", "left")
                     .select("referral_date", "referrer", "referred")
                     .orderBy("referral_date"))

etl(referrals, customers).show()
```

</details>

<details>
<summary><b>Q7</b> &nbsp;Show the pairs of customers who live in the same city: city, customer_a and customer_b (names). Each pair ...</summary>

```python
def etl(customers):
    a, b = customers.alias("a"), customers.alias("b")
    return (a.join(b, (F.col("a.city") == F.col("b.city")) & (F.col("a.customer_id") < F.col("b.customer_id")))
             .select(F.col("a.city").alias("city"), F.col("a.name").alias("customer_a"), F.col("b.name").alias("customer_b"))
             .orderBy("city"))

etl(customers).show()
```

</details>

<details>
<summary><b>Q8</b> &nbsp;For each delivered order, find the promotion of its product's category that was running on the order date. ...</summary>

```python
def etl(orders, products, promotions):
    return (orders.filter(F.col("status") == "delivered")
                  .join(products, "product_id")
                  .join(promotions, (products.category == promotions.category)
                                    & F.col("order_date").between(F.col("start_date"), F.col("end_date")))
                  .select("order_id", "product_name", "promo_id", "discount_pct")
                  .orderBy("order_id"))

etl(orders, products, promotions).show()
```

</details>

<details>
<summary><b>Q9</b> &nbsp;A window partitioned by customer_id: which stage is skewed, why doesn't AQE fix it, and what would you do?</summary>

The window's stage (200 tasks): a median around 0.02 s and a slowest task around 3 s, over 100 times the median. That task holds customer 1's partition: 3 million rows to receive, sort and number, while every other task has about 35,000.

AQE can't fix it: its skew handling only splits the partitions of a **join**. A window needs every row of a key in one task, in order, so the partition of one key can't be split.

For a daily job: (1) check whether the reseller belongs in this report at all, and if not, filter it out before the window; (2) if you only need one value per customer (a count, a total, the first date), use `groupBy`: partial aggregation makes the hot key cheap (the last cell of Lesson 5); (3) otherwise give the hot key its own path (sort its 3 million rows with `orderBy`, which spreads them over many tasks by range), run the window on the other customers, and `union` the two; (4) at least give the tasks enough memory not to spill.

</details>

<details>
<summary><b>Q10</b> &nbsp;Add up the items of each city, sorted by city. In this cell Spark can't broadcast and AQE's skew join is off ...</summary>

```python
def etl(sales, customers_big):
    salts = 16
    # big side: customer 1's rows get a random salt 0..15, every other row gets 0
    salted_sales = sales.withColumn(
        "salt", F.when(F.col("customer_id") == 1, (F.rand(42) * salts).cast("int")).otherwise(0))
    # other side: customer 1's row is copied once per salt value, every other row once with salt 0
    salted_customers = customers_big.withColumn("salt", F.explode(
        F.when(F.col("customer_id") == 1, F.array(*[F.lit(i) for i in range(salts)]))
         .otherwise(F.array(F.lit(0)))))
    return (salted_sales.join(salted_customers, ["customer_id", "salt"])
                        .groupBy("city").agg(F.sum("quantity").alias("items"))
                        .orderBy("city"))

etl(sales, customers_big).show()
```

The join key is now (customer_id, salt). Customer 1's 3 million sales are spread over 16 salt values, about 190,000 rows each, which hash into different partitions, and each of those partitions finds its copy of customer 1's row. Every other customer has salt 0 on both sides, so it joins as before, and the totals don't change. Only the hot key is copied (16 rows instead of 1); salting every key would copy all 2 million customers 16 times. The gain is smaller than 16 times because the 16 salted keys are hashed into 200 partitions, and two or three of them often land in the same one. With only `customer_id` and `quantity` shuffled from sales, the reseller's partition is about 10 MB compressed, so even with lowered thresholds AQE wouldn't treat it as skewed (Lesson 5): this is the kind of case where salting by hand is still needed. In real life, try `F.broadcast` first.

Why the total time hardly changes here: with 4 cores, the other 199 tasks keep three cores busy while the hot task runs anyway. On a cluster, where hundreds of cores wait for one straggler, the slowest task decides the stage's time, and that's what salting fixes.

</details>

<details>
<summary><b>Q11</b> &nbsp;Show product_name, category, price and diff_from_avg (the price minus its category's average price, 2 ...</summary>

```python
def etl(products):
    w = W.partitionBy("category")
    return (products.select("product_name", "category", "price",
                            F.round(F.col("price") - F.avg("price").over(w), 2).alias("diff_from_avg"))
                    .orderBy("category", "price"))

etl(products).show()
```

</details>

<details>
<summary><b>Q12</b> &nbsp;For the orders of customers 1 and 2, show order_id, customer_id, quantity and pct_of_customer: the order's ...</summary>

```python
def etl(orders):
    w = W.partitionBy("customer_id")
    return (orders.filter(F.col("customer_id").isin(1, 2))
                  .select("order_id", "customer_id", "quantity",
                          F.round(F.col("quantity") / F.sum("quantity").over(w) * 100, 1).alias("pct_of_customer"))
                  .orderBy("order_id"))

etl(orders).show()
```

</details>

<details>
<summary><b>Q13</b> &nbsp;Show the most expensive product of each category: category, product_name and price, sorted by category.</summary>

```python
def etl(products):
    w = W.partitionBy("category").orderBy(F.col("price").desc())
    return (products.withColumn("rn", F.row_number().over(w))
                    .filter(F.col("rn") == 1)
                    .select("category", "product_name", "price")
                    .orderBy("category"))

etl(products).show()
```

</details>

<details>
<summary><b>Q14</b> &nbsp;Show the top 2 products of each category by items delivered (on a tie, the smaller product_id first). ...</summary>

```python
def etl(orders, products):
    items = (orders.filter(F.col("status") == "delivered")
                   .join(products, "product_id")
                   .groupBy("category", "product_id", "product_name")
                   .agg(F.sum("quantity").alias("items")))
    w = W.partitionBy("category").orderBy(F.col("items").desc(), "product_id")
    return (items.withColumn("rank", F.row_number().over(w))
                 .filter(F.col("rank") <= 2)
                 .select("category", "product_name", "items", "rank")
                 .orderBy("category", "rank"))

etl(orders, products).show()
```

</details>

<details>
<summary><b>Q15</b> &nbsp;For every order of customers 1 and 3, show how many days passed since the same customer's previous order ...</summary>

```python
def etl(orders):
    w = W.partitionBy("customer_id").orderBy("order_date", "order_id")
    return (orders.filter(F.col("customer_id").isin(1, 3))
                  .withColumn("days_since_prev", F.datediff("order_date", F.lag("order_date").over(w)))
                  .select("customer_id", "order_id", "order_date", "days_since_prev")
                  .orderBy("customer_id", "order_date", "order_id"))

etl(orders).show()
```

</details>

<details>
<summary><b>Q16</b> &nbsp;A running total: for customers 1 and 2, show customer_id, order_id, order_date, quantity and items_so_far ...</summary>

```python
def etl(orders):
    w = (W.partitionBy("customer_id").orderBy("order_date", "order_id")
          .rowsBetween(W.unboundedPreceding, W.currentRow))
    return (orders.filter(F.col("customer_id").isin(1, 2))
                  .withColumn("items_so_far", F.sum("quantity").over(w))
                  .select("customer_id", "order_id", "order_date", "quantity", "items_so_far")
                  .orderBy("customer_id", "order_date", "order_id"))

etl(orders).show()
```

</details>

<details>
<summary><b>Q17</b> &nbsp;In each city, find the customer with the most orders (on a tie, the smaller customer_id). Customers without a ...</summary>

```python
def etl(orders, customers):
    counts = orders.groupBy("customer_id").agg(F.count("*").alias("orders"))
    w = W.partitionBy("city").orderBy(F.desc("orders"), "customer_id")
    return (customers.filter(F.col("city").isNotNull())
                     .join(counts, "customer_id")
                     .withColumn("rn", F.row_number().over(w))
                     .filter(F.col("rn") == 1)
                     .select("city", "name", "orders")
                     .orderBy("city"))

etl(orders, customers).show()
```

</details>

<details>
<summary><b>Q18</b> &nbsp;For each customer with at least 2 orders, find the longest gap in days between two consecutive orders. ...</summary>

```python
def etl(orders):
    w = W.partitionBy("customer_id").orderBy("order_date", "order_id")
    return (orders.filter(F.col("customer_id").isNotNull())
                  .withColumn("gap_days", F.datediff("order_date", F.lag("order_date").over(w)))
                  .groupBy("customer_id")
                  .agg(F.max("gap_days").alias("longest_gap_days"))
                  .filter(F.col("longest_gap_days").isNotNull())
                  .orderBy("customer_id"))

etl(orders).show()
```

A customer with one order has only a null gap, and `max` of nulls is null: that's how the filter removes them.

</details>

<details>
<summary><b>Q19</b> &nbsp;The top 3 products of each category by items sold: category, rank (1 to 3; on a tie, the smaller product_id ...</summary>

```python
def etl(sales, products_big):
    per_product = sales.groupBy("product_id").agg(F.sum("quantity").alias("items"))
    w = W.partitionBy("category").orderBy(F.desc("items"), "product_id")
    return (per_product.join(products_big, "product_id")
                       .withColumn("rank", F.row_number().over(w))
                       .filter(F.col("rank") <= 3)
                       .select("category", "rank", "product_id", "product_name", "items")
                       .orderBy("category", "rank"))

etl(sales, products_big).show(30)
```

Mistake 1: `distinct()` "just in case". The rows are already unique (sale_id), but Spark doesn't know that, so it shuffles all 10 million joined rows with every column to compare them (about 500 MB, most of the spill). Mistake 2: joining before aggregating. The join makes 10 million wide rows, and the `groupBy` then hashes the text columns `category` and `product_name` for each of them. Aggregating `sales` by `product_id` first shrinks 10 million rows to 50,000 inside each task (partial aggregation), and only then does the broadcast join add the names. The window at the end is cheap either way: it ranks 50,000 rows, with a `WindowGroupLimit` before its shuffle.

</details>